In [ ]:
import sys

from pathlib import Path

module_path = str(Path("..").resolve())
if module_path not in sys.path:
    sys.path.append(module_path)

import utils.data_io as data_io
import utils.graph_stats as graph_stats
import utils.graph_utils as graph_utils
import utils.vis as vis

from config_handlers.sk_handler import SKHandler

# Main

In [2]:
sk_hdl = SKHandler()
DATASETS = sk_hdl.get_ds_list()
DATASETS.remove("vqav2_holdout")
base_dir = Path("..")
# need to add <ds_name>/<file_name>

all_datasets = DATASETS * 3
all_datasets.sort()

In [3]:
# Column mapping
og_cols = ("from_concept", "to_concept", "relationship")
new_cols = ("new_from_concept", "new_to_concept", "new_relationship")

# cols = [new_cols if '_val.csv' in dp else og_cols for dp in data_paths]
# cols = [new_cols if s == "val" else og_cols for s in all_steps]

# Loading concept and predicate frequencies from VG1800 dataset.
# VG1800 is a subsample of VG made by the authors of IETrans.
vg1800_dict = graph_utils.load_vg1800_dict()

In [4]:
def get_sk_data() -> dict:
    all_sks = {}

    for ds in DATASETS:
        sk_hdl.set_curr_ds(ds)
        if ds not in all_sks:
            all_sks[ds] = {}

        for ds_class in sk_hdl.get_classes():
            sk_hdl.set_curr_class(ds_class)
            sk_path = base_dir.joinpath(sk_hdl.get_sk_final_path())
            sk_data = data_io.load_jsonl(sk_path)

            if ds_class not in all_sks[ds]:
                all_sks[ds][ds_class] = {}

            sk_dict = {}
            for curr_sample in sk_data:
                question_id = curr_sample["question_id"]
                if question_id not in sk_dict:
                    sk_dict[question_id] = curr_sample["relations"]
            all_sks[ds][ds_class] = sk_dict

    return all_sks

In [5]:
def get_sk_details(sks: dict) -> dict:
    data = {
        "graphs": {ds: {} for ds in DATASETS},
        "summaries": {ds: {} for ds in DATASETS},
        "sample_stats": {ds: {} for ds in DATASETS},
        "count_concepts": {ds: {} for ds in DATASETS},
        "count_preds": {ds: {} for ds in DATASETS},
    }

    for ds in sks.keys():
        print(f"Processing {ds}")
        sk_hdl.set_curr_ds(ds)

        for ds_class in sks[ds].keys():
            sk_hdl.set_curr_class(ds_class)

            # No distinction between classes, put everything from 1 dataset together
            for sk_idx, sk_data in sks[ds][ds_class].items():
                data["graphs"][ds][sk_idx] = graph_utils.sk_to_nx(sk_data)

            sample_stats, stats_summary = graph_stats.compute_stats(
                data["graphs"][ds], vg1800_dict
            )

        # Update
        data["sample_stats"][ds] = sample_stats
        data["summaries"][ds] = stats_summary
        data["count_concepts"][ds] = stats_summary["count_concepts"]
        data["count_preds"][ds] = stats_summary["count_preds"]

    return data

In [6]:
# Load SK data
sks = get_sk_data()
# Process SK data
sk_data = get_sk_details(sks)

Processing llava-bench
Processing mmbench
Processing seed
Processing vqav2


In [7]:
sk_data.keys()

dict_keys(['graphs', 'summaries', 'sample_stats', 'count_concepts', 'count_preds'])

In [8]:
sk_data["summaries"].keys()

dict_keys(['llava-bench', 'mmbench', 'seed', 'vqav2'])

In [9]:
(sk_data["summaries"]["mmbench"].keys())

dict_keys(['count_concepts', 'avg_concepts', 'std_concepts', 'avg_ic_concepts', 'std_ic_concepts', 'avg_h_concepts', 'std_h_concepts', 'count_preds', 'avg_preds', 'std_preds', 'avg_ic_preds', 'std_ic_preds', 'avg_h_preds', 'std_h_preds', 'avg_ic_graph', 'std_ic_graph', 'avg_h_graph', 'std_h_graph'])

In [11]:
for k, v in sk_data["summaries"]["mmbench"].items():
    print(f"{k.upper()}: {v:.2f}")

COUNT_CONCEPTS: 1614.00
AVG_CONCEPTS: 11.61
STD_CONCEPTS: 5.20
AVG_IC_CONCEPTS: 16.37
STD_IC_CONCEPTS: 2.39
AVG_H_CONCEPTS: 0.03
STD_H_CONCEPTS: 0.01
COUNT_PREDS: 1078.00
AVG_PREDS: 7.76
STD_PREDS: 3.95
AVG_IC_PREDS: 11.23
STD_IC_PREDS: 2.75
AVG_H_PREDS: 0.09
STD_H_PREDS: 0.08
AVG_IC_GRAPH: 13.80
STD_IC_GRAPH: 1.73
AVG_H_GRAPH: 0.06
STD_H_GRAPH: 0.04


# Save results to disk

Save statistics

In [ ]:
out_dir = Path("..", "data", "stats")
data_io.make_dir(out_dir)

for ds in DATASETS:
    print(f"Saving {ds}")

    # Summaries
    summary_file = out_dir.joinpath(ds, "summary.json")
    data_io.make_dir(summary_file.parent)
    data_io.save_json(sk_data["summaries"][ds], summary_file)
    # Sample-level
    sample_file = out_dir.joinpath(ds, "samples.json")
    data_io.save_json(sk_data["sample_stats"][ds], sample_file)

Save statistics graphs

In [ ]:
# Create output dir for charts
charts_dir = Path(".", "stats_charts")
data_io.make_dir(charts_dir)

vis.plot_boxplot_sk(
    sk_data["count_concepts"], sk_data["count_preds"], charts_dir.joinpath("box_sk.py")
)

# Old

To save as Excel files

In [ ]:
# # Save to Excel with two sheets
# sample_stats_df = pd.DataFrame(sample_stats)
# dataset_stats_df = pd.DataFrame([dataset_stats])
# output_path = f"output/{step}/graph_statistics.xlsx"
# with pd.ExcelWriter(output_path) as writer:
#     sample_stats_df.to_excel(writer, sheet_name="Sample Statistics", index=False)
#     dataset_stats_df.to_excel(writer, sheet_name="Dataset Summary", index=False)